In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
import time

In [2]:
# 1. Load & Preprocess
df1 = pd.read_csv('all_songs_data_50k.csv')
df2 = pd.read_csv('validation_dataset.csv')
# Combine datasets
df = pd.concat([df1, df2], ignore_index=True)

In [3]:
df

,_id,recipient,message,song_id,song_name,song_artist,song_image,created_at
0,6853b58d17e1fe68eff7d1bf,mark,"Hi, uhm, yeah, everytime na tingnan ko yung re...",2U3jOPfO4wZZFaaWS4Dcj6,stranger,Olivia Rodrigo,https://i.scdn.co/image/ab67616d0000b2734063d6...,2025-06-19T07:00:29.319655884Z
1,6853b58717e1fe68eff7d1be,dimas ali,I don't know if you still care about me or not...,6BQPRUFGHsgtNYJxuwhktR,This Is How It Feels (with Laufey),"d4vd, Laufey",https://i.scdn.co/image/ab67616d0000b27364fa1b...,2025-06-19T07:00:23.212049427Z
2,6853b58417e1fe68eff7d1bd,2229,"I'll always love you from this day on, now and...",37mbA1RwYja7WFaIOzSczX,You'll Be In My Heart - Spotify Singles,NIKI,https://i.scdn.co/image/ab67616d0000b273e68520...,2025-06-19T07:00:20.622764007Z
3,6853b56717e1fe68eff7d1bc,savira ramadhani m,.,6GCW5Muk3u0cM5QTkS4C9a,The One That Got Away,Katy Perry,https://i.scdn.co/image/ab67616d0000b27343219f...,2025-06-19T06:59:51.325948562Z
4,6853b56617e1fe68eff7d1bb,drix,"hi drix, i hope when out paths met again, sana...",4cBm8rv2B5BJWU2pDaHVbF,Multo,Cup of Joe,https://i.scdn.co/image/ab67616d0000b273394048...,2025-06-19T06:59:50.396334841Z
...,...,...,...,...,...,...,...,...
53534,68138f211fc54584f668c3bf,ayu,hey! i've seen u once sa morato playing billia...,6cEguiQecbXrFlsnMi2ysr,Come and See Me (feat. Drake),"PARTYNEXTDOOR, Drake",https://i.scdn.co/image/ab67616d0000b273240b49...,2025-05-01T15:11:29.050072684Z
53535,6812cf9b1fc54584f668a368,ayu,untuk seseorang di bawah yg menggunakan lagu i...,57RA3JGafJm5zRtKJiKPIm,Are You Bored Yet? (feat. Clairo),"Wallows, Clairo",https://i.scdn.co/image/ab67616d0000b27384feca...,2025-05-01T01:34:19.555554536Z
53536,680bd9c01fc54584f6678c2e,ayu,disappointed. thanks a lot,0bht8SpPHXiUMApKcua4Mz,Take Care,NIKI,https://i.scdn.co/image/ab67616d0000b2734d3d39...,2025-04-25T18:51:44.276404573Z
53537,680b59f31fc54584f6676859,ayu,jujur aja guel sedih liat org yang gue cintai ...,1tqPw8Hf88h24Bpt2SzuYE,Di Akhir Perang,Nadin Amizah,https://i.scdn.co/image/ab67616d0000b273948476...,2025-04-25T09:46:27.779280549Z


In [4]:
# Karena song_id berupa string, kita perlu membuat mapping ke integer
print("Membuat mapping song_id string ke integer...")

# Ambil unique song_id dan buat mapping
unique_song_ids = df['song_id'].unique()
song_id_to_int = {song_id: i for i, song_id in enumerate(unique_song_ids)}
int_to_song_id = {i: song_id for song_id, i in song_id_to_int.items()}

# Tambahkan kolom song_id_int ke dataframe
df['song_id_int'] = df['song_id'].map(song_id_to_int)

print(f"Unique song_ids: {len(unique_song_ids)}")
print(f"Contoh mapping: {list(song_id_to_int.items())[:5]}")
print(f"Range song_id_int: 0 - {max(song_id_to_int.values())}")

Membuat mapping song_id string ke integer...
Unique song_ids: 4377
Contoh mapping: [('2U3jOPfO4wZZFaaWS4Dcj6', 0), ('6BQPRUFGHsgtNYJxuwhktR', 1), ('37mbA1RwYja7WFaIOzSczX', 2), ('6GCW5Muk3u0cM5QTkS4C9a', 3), ('4cBm8rv2B5BJWU2pDaHVbF', 4)]
Range song_id_int: 0 - 4376


In [5]:
# --- Konfigurasi Model & Pelatihan ---
CONFIG = {
    # Dimensi Embedding untuk Teks
    "text_embedding_dim": 150, # dinaikkan
    "hidden_dim": 300, # dinaikkan
    "n_layers": 4,
    "dropout": 0.5, # disesuaikan
    # Dimensi Output Embedding (HARUS SAMA untuk kedua menara)
    "output_embedding_dim": 64,
    "batch_size": 128,
    "learning_rate": 0.001,
    "epochs": 6, # dinaikkan untuk konvergensi yang lebih baik
    "max_len": 50,
    "margin": 0.2,
    "min_song_freq": 13, # Batas frekuensi minimum untuk lagu
}

In [6]:
# --- 1. Persiapan Device (CPU/GPU) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU terdeteksi: {torch.cuda.get_device_name(0)}. Pelatihan akan menggunakan GPU.")
else:
    device = torch.device("cpu")
    print("GPU tidak ditemukan. Pelatihan akan menggunakan CPU.")

GPU terdeteksi: NVIDIA GeForce GTX 1650 Ti. Pelatihan akan menggunakan GPU.


In [7]:
# --- 2. Pra-pemrosesan Data ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_vocab(all_texts):
    word_counts = Counter(word for text in all_texts for word in text.split())
    vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common())}
    vocab['<PAD>'] = 0
    vocab['<UNK>'] = 1
    return vocab

In [8]:
# --- 3. Architecture Model ---

# Tower Pertama(message)
class MessageTower(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["text_embedding_dim"], padding_idx=0)
        self.gru = nn.GRU(
            config["text_embedding_dim"],
            config["hidden_dim"],
            num_layers=config["n_layers"],
            dropout=config["dropout"],
            batch_first=True
        )
        # Layer terakhir untuk memproyeksikan output GRU ke dimensi embedding akhir
        self.fc = nn.Linear(config["hidden_dim"], config["output_embedding_dim"])

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = hidden[-1] # Ambil hidden state terakhir dari layer terakhir
        # Normalisasi L2 pada output embedding adalah praktik yang baik
        return F.normalize(self.fc(hidden), p=2, dim=1)

# Tower Kedua(song)
class SongTower(nn.Module):
    def __init__(self, max_song_int_id, config):
        super().__init__()
        # Gunakan max_song_int_id + 1 untuk mengakomodasi semua ID lagu integer
        self.embedding = nn.Embedding(max_song_int_id + 1, config["output_embedding_dim"])

    def forward(self, x):
        # Normalisasi L2 pada output embedding
        return F.normalize(self.embedding(x), p=2, dim=1)

In [9]:
# --- 4. Dataset dan DataLoader ---
class RecommendationDataset(Dataset):
    def __init__(self, messages, song_ids, vocab, max_len):
        self.messages = messages
        self.song_ids = song_ids # Berisi ID lagu (integer)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, idx):
        # Ambil pasangan (pesan, lagu) yang positif
        message = self.messages[idx]
        song_id = self.song_ids[idx]
        
        # Vectorize pesan
        vectorized_msg = [self.vocab.get(word, self.vocab['<UNK>']) for word in message.split()]
        if len(vectorized_msg) < self.max_len:
            vectorized_msg += [self.vocab['<PAD>']] * (self.max_len - len(vectorized_msg))
        else:
            vectorized_msg = vectorized_msg[:self.max_len]
        
        return torch.tensor(vectorized_msg, dtype=torch.long), torch.tensor(song_id, dtype=torch.long)

In [10]:
# --- 5. Fungsi Evaluasi ---
def evaluate_model(message_tower, song_tower, data_loader, criterion, device):
    """Evaluasi model pada dataset validation atau test."""
    message_tower.eval()
    song_tower.eval()
    
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for messages, positive_songs in data_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            
            # Dapatkan embeddings
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            
            # Hitung loss (sama seperti training)
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            loss = loss_pos + loss_neg
            
            total_loss += loss.item()
            
            # Hitung accuracy berdasarkan cosine similarity
            similarities = F.cosine_similarity(message_embeddings, song_embeddings)
            negative_similarities = F.cosine_similarity(message_embeddings, negative_song_embeddings)
            
            # Prediksi benar jika similarity positif > similarity negatif
            correct = (similarities > negative_similarities).sum().item()
            correct_predictions += correct
            total_samples += messages.size(0)
    
    avg_loss = total_loss / len(data_loader)
    accuracy = correct_predictions / total_samples
    
    return avg_loss, accuracy

# --- 6. Fungsi Pelatihan dengan Evaluasi ---
def train_recommendation_model(message_tower, song_tower, train_loader, val_loader, config, device):
    message_tower.to(device)
    song_tower.to(device)
    
    # Gabungkan parameter dari kedua model untuk optimizer
    optimizer = optim.AdamW(
        list(message_tower.parameters()) + list(song_tower.parameters()),
        lr=config["learning_rate"]
    )
    criterion = nn.CosineEmbeddingLoss(margin=config["margin"])
    
    # Tracking untuk best model
    best_val_loss = float('inf')
    best_epoch = 0
    
    print("🚀 Memulai pelatihan model...")
    print("-" * 80)
    
    for epoch in range(1, config["epochs"] + 1):
        start_time = time.time()
        
        # === TRAINING ===
        message_tower.train()
        song_tower.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for messages, positive_songs in train_loader:
            messages, positive_songs = messages.to(device), positive_songs.to(device)
            
            optimizer.zero_grad()
            
            # Dapatkan embeddings
            message_embeddings = message_tower(messages)
            song_embeddings = song_tower(positive_songs)
            
            # Loss calculation
            loss_pos = criterion(message_embeddings, song_embeddings, torch.ones(messages.size(0)).to(device))
            negative_song_embeddings = torch.roll(song_embeddings, shifts=1, dims=0)
            loss_neg = criterion(message_embeddings, negative_song_embeddings, -torch.ones(messages.size(0)).to(device))
            loss = loss_pos + loss_neg
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Accuracy calculation
            similarities = F.cosine_similarity(message_embeddings, song_embeddings)
            negative_similarities = F.cosine_similarity(message_embeddings, negative_song_embeddings)
            correct = (similarities > negative_similarities).sum().item()
            train_correct += correct
            train_total += messages.size(0)
        
        # === VALIDATION ===
        val_loss, val_accuracy = evaluate_model(message_tower, song_tower, val_loader, criterion, device)
        
        # Metrics
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = train_correct / train_total
        end_time = time.time()
        
        # Print progress
        print(f"Epoch {epoch:2d}/{config['epochs']} | "
              f"Time: {end_time - start_time:5.1f}s | "
              f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            # Simpan best model (opsional)
        elif epoch - best_epoch > 5:  # Early stopping setelah 5 epoch tanpa improvement
            print(f"⚠️  Early stopping triggered. Best epoch: {best_epoch}")
            break
    
    print("-" * 80)
    print(f"✅ Pelatihan selesai! Best validation loss: {best_val_loss:.4f} pada epoch {best_epoch}")

In [11]:
# --- 7. Fungsi untuk Rekomendasi (Inference) ---
def build_song_index(song_tower, all_song_int_ids, device):
    """Proses semua lagu untuk membuat matriks embedding (indeks)."""
    print("\nMembangun indeks embedding untuk semua lagu...")
    song_tower.eval()
    with torch.no_grad():
        # Buat tensor berisi semua ID lagu integer
        song_ids_tensor = torch.tensor(all_song_int_ids).to(device)
        # Dapatkan embedding untuk semua lagu sekaligus
        song_index_embeddings = song_tower(song_ids_tensor)
    print("Indeks berhasil dibangun.")
    return song_index_embeddings

def recommend_songs(message_text, message_tower, song_index, all_song_int_ids, songint_to_name, songint_to_original_id, vocab, config, device, top_k=10):
    """Memberikan top_k rekomendasi lagu untuk sebuah pesan."""
    message_tower.eval()
    
    # 1. Proses pesan input menjadi embedding
    cleaned_text = clean_text(message_text)
    vectorized = [vocab.get(word, vocab['<UNK>']) for word in cleaned_text.split()]
    if len(vectorized) < config["max_len"]:
        vectorized += [vocab['<PAD>']] * (config["max_len"] - len(vectorized))
    else:
        vectorized = vectorized[:config["max_len"]]
        
    query_tensor = torch.tensor(vectorized, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        query_embedding = message_tower(query_tensor)
        
    # 2. Hitung kemiripan (cosine similarity) antara query dan semua lagu di indeks
    similarities = F.cosine_similarity(query_embedding, song_index)
    
    # 3. Dapatkan top_k lagu dengan similarity tertinggi
    top_results = torch.topk(similarities, k=top_k)
    
    # 4. Kembalikan song_id asli (string), nama lagu dan skornya
    recommendations = []
    for score, idx in zip(top_results.values, top_results.indices):
        song_int_id = all_song_int_ids[idx.item()]  # Dapatkan song_id integer dari list
        original_song_id = songint_to_original_id[song_int_id]  # Konversi ke song_id asli (string)
        song_name = songint_to_name[song_int_id]  # Dapatkan nama lagu
        recommendations.append({
            "song_id": original_song_id,  # Song ID asli (string)
            "song_name": song_name,
            "similarity_score": score.item()
        })
    return recommendations

# MAIN EXECUTION


# ✨ Peningkatan Model

## 🔄 Perubahan yang Diimplementasikan:

### 1. **Data Splitting**

- ✅ Train: 70% dari data
- ✅ Validation: 15% dari data
- ✅ Test: 15% dari data
- ✅ Stratified splitting untuk distribusi label yang seimbang

### 2. **Sistem Evaluasi Komprehensif**

- ✅ **Training Loss & Accuracy**: Monitoring selama pelatihan
- ✅ **Validation Loss & Accuracy**: Evaluasi setiap epoch
- ✅ **Test Loss & Accuracy**: Evaluasi final pada data unseen

### 3. **Fitur Tambahan**

- ✅ **Early Stopping**: Mencegah overfitting
- ✅ **Best Model Tracking**: Menyimpan performa terbaik
- ✅ **Progress Monitoring**: Output yang lebih informatif
- ✅ **Accuracy Metric**: Berdasarkan cosine similarity comparison

### 4. **Song ID Management**

- ✅ Menggunakan **song_id asli** dari dataset (bukan index mapping)
- ✅ Mapping otomatis antara string song_id dan integer untuk model
- ✅ Output rekomendasi menampilkan song_id asli


In [12]:
# --- Pra-pemrosesan Data Asli ---
# Hapus baris dengan nilai kosong di kolom penting
df.dropna(subset=['message', 'song_name', 'song_artist'], inplace=True)
# Hapus pesan atau nama lagu yang hanya berisi spasi
df = df[df['message'].str.strip() != '']
df = df[df['song_name'].str.strip() != '']
df.reset_index(drop=True, inplace=True)

# Filter lagu yang jarang muncul untuk mengurangi noise
song_counts = df['song_id_int'].value_counts()
frequent_song_ids = song_counts[song_counts >= CONFIG["min_song_freq"]].index
df = df[df['song_id_int'].isin(frequent_song_ids)].reset_index(drop=True)

print(f"Dataset dimuat dan dibersihkan. Jumlah data setelah filtering: {len(df)}")

df['cleaned_message'] = df['message'].apply(clean_text)

Dataset dimuat dan dibersihkan. Jumlah data setelah filtering: 42053


In [13]:
# 2. Build Vocab dan Song Mapping
vocab = build_vocab(df['cleaned_message'])

# Gunakan song_id_int untuk model (karena embedding layer butuh integer)
# Tapi simpan mapping ke song_id asli (string)
all_song_int_ids = df['song_id_int'].unique().tolist()
max_song_int_id = max(all_song_int_ids)

# Buat mapping untuk mendapatkan info lagu dari song_id_int
songint_to_name = dict(zip(df['song_id_int'], df['song_name']))
songint_to_original_id = dict(zip(df['song_id_int'], df['song_id']))

# Buat dataframe lagu unik untuk reference
song_df = df[['song_id', 'song_id_int', 'song_name', 'song_artist']].drop_duplicates().reset_index(drop=True)

vocab_size = len(vocab)
num_songs = len(all_song_int_ids)
print(f"\nUkuran Vocabulary: {vocab_size}, Jumlah Lagu Unik: {num_songs}")
print(f"Range Song ID Integer: 0 - {max_song_int_id}")


Ukuran Vocabulary: 77707, Jumlah Lagu Unik: 638
Range Song ID Integer: 0 - 3477


In [14]:
# 3. Data Splitting & Create DataLoader
X = df['cleaned_message'].values
y = df['song_id_int'].values

# Split data: 70% train, 15% validation, 15% test
print("Membagi data menjadi train, validation, dan test set...")
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples") 
print(f"Test set: {len(X_test)} samples")

# Create datasets and dataloaders
train_dataset = RecommendationDataset(X_train, y_train, vocab, CONFIG["max_len"])
val_dataset = RecommendationDataset(X_val, y_val, vocab, CONFIG["max_len"])
test_dataset = RecommendationDataset(X_test, y_test, vocab, CONFIG["max_len"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

Membagi data menjadi train, validation, dan test set...
Training set: 29437 samples
Validation set: 6308 samples
Test set: 6308 samples


In [15]:
# 4. Inisialisasi Model
message_tower = MessageTower(vocab_size, CONFIG)
song_tower = SongTower(max_song_int_id, CONFIG)

In [16]:
# 5. Latih Model dengan Evaluasi
train_recommendation_model(message_tower, song_tower, train_loader, val_loader, CONFIG, device)

🚀 Memulai pelatihan model...
--------------------------------------------------------------------------------
Epoch  1/6 | Time:  21.9s | Train Loss: 0.8794 | Train Acc: 0.5045 | Val Loss: 0.8446 | Val Acc: 0.5182
Epoch  1/6 | Time:  21.9s | Train Loss: 0.8794 | Train Acc: 0.5045 | Val Loss: 0.8446 | Val Acc: 0.5182
Epoch  2/6 | Time:  21.7s | Train Loss: 0.8258 | Train Acc: 0.5225 | Val Loss: 0.8197 | Val Acc: 0.5136
Epoch  2/6 | Time:  21.7s | Train Loss: 0.8258 | Train Acc: 0.5225 | Val Loss: 0.8197 | Val Acc: 0.5136
Epoch  3/6 | Time:  21.8s | Train Loss: 0.8018 | Train Acc: 0.5368 | Val Loss: 0.7924 | Val Acc: 0.5444
Epoch  3/6 | Time:  21.8s | Train Loss: 0.8018 | Train Acc: 0.5368 | Val Loss: 0.7924 | Val Acc: 0.5444
Epoch  4/6 | Time:  21.9s | Train Loss: 0.7782 | Train Acc: 0.5581 | Val Loss: 0.7744 | Val Acc: 0.5566
Epoch  4/6 | Time:  21.9s | Train Loss: 0.7782 | Train Acc: 0.5581 | Val Loss: 0.7744 | Val Acc: 0.5566
Epoch  5/6 | Time:  21.9s | Train Loss: 0.7682 | Train Acc

In [17]:
# 7. Bangun Indeks untuk Inference
song_index = build_song_index(song_tower, all_song_int_ids, device)


Membangun indeks embedding untuk semua lagu...
Indeks berhasil dibangun.


In [18]:
# 6. Evaluasi pada Test Set
print("\n🧪 Evaluasi performa final pada test set...")
criterion = nn.CosineEmbeddingLoss(margin=CONFIG["margin"])
test_loss, test_accuracy = evaluate_model(message_tower, song_tower, test_loader, criterion, device)

print(f"📊 Test Set Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   Total Test Samples: {len(test_dataset)}")

# Simpan hasil evaluasi
eval_results = {
    'test_loss': test_loss,
    'test_accuracy': test_accuracy,
    'test_samples': len(test_dataset)
}


🧪 Evaluasi performa final pada test set...
📊 Test Set Results:
   Test Loss: 0.7563
   Test Accuracy: 0.5720 (57.20%)
   Total Test Samples: 6308
📊 Test Set Results:
   Test Loss: 0.7563
   Test Accuracy: 0.5720 (57.20%)
   Total Test Samples: 6308


In [19]:
# 8. Tes Rekomendasi
print("\n--- Tes Rekomendasi ---")
test_message_1 = "masih inget aku??"
test_message_2 = "i love you more than anything in the world"

recs_1 = recommend_songs(test_message_1, message_tower, song_index, all_song_int_ids, songint_to_name, songint_to_original_id, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_1}'")
for rec in recs_1:
   print(f"  - ID: {rec['song_id']} | {rec['song_name']} (Score: {rec['similarity_score']:.4f})")
   
recs_2 = recommend_songs(test_message_2, message_tower, song_index, all_song_int_ids, songint_to_name, songint_to_original_id, vocab, CONFIG, device)
print(f"\nRekomendasi untuk pesan: '{test_message_2}'")
for rec in recs_2:
   print(f"  - ID: {rec['song_id']} | {rec['song_name']} (Score: {rec['similarity_score']:.4f})")


--- Tes Rekomendasi ---

Rekomendasi untuk pesan: 'masih inget aku??'
  - ID: 2U3jOPfO4wZZFaaWS4Dcj6 | stranger (Score: 0.6713)
  - ID: 0ROj512WvJ1eqeELd7MEdJ | Sampai Jadi Debu (Menampilkan Gardika Gigih) (Score: 0.4798)
  - ID: 4MagTPnkPiDuIa4P8GtW1b | I Like Me Better (Score: 0.4492)
  - ID: 2262bWmqomIaJXwCRHr13j | Sailor Song (Score: 0.4276)
  - ID: 4cBm8rv2B5BJWU2pDaHVbF | Multo (Score: 0.4271)
  - ID: 5GoY2ioRnfQayqsNx4HaXh | Anaheim (Score: 0.4137)
  - ID: 1fDFHXcykq4iw8Gg7s5hG9 | About You (Score: 0.4108)
  - ID: 34xTFwjPQ1dC6uJmleno7x | Godspeed (Score: 0.4065)
  - ID: 7gB98EWp4p9VKxOToaqrhM | Cinta Terakhir (Score: 0.4061)
  - ID: 2IVsRhKrx8hlQBOWy4qebo | Mr. Loverman (Score: 0.4038)

Rekomendasi untuk pesan: 'i love you more than anything in the world'
  - ID: 7D0RhFcb3CrfPuTJ0obrod | Sparks (Score: 0.5614)
  - ID: 3X9c4tBzSdGhlO4Fx3WYgW | Lifetime (Score: 0.5306)
  - ID: 6Qyc6fS4DsZjB2mRW9DsQs | Iris (Score: 0.4930)
  - ID: 4cBm8rv2B5BJWU2pDaHVbF | Multo (Score: 0.4879)
 

In [21]:
# Ekspor model setelah training
print("Mengekspor model...")
torch.save({
    'message_tower_state_dict': message_tower.state_dict(),
    'song_tower_state_dict': song_tower.state_dict(),
    'vocab': vocab,
    'all_song_int_ids': all_song_int_ids,
    'songint_to_name': songint_to_name,
    'songint_to_original_id': songint_to_original_id,
    'max_song_int_id': max_song_int_id,
    'song_id_to_int': song_id_to_int,
    'int_to_song_id': int_to_song_id,
    'CONFIG': CONFIG,
    'vocab_size': vocab_size,
    'num_songs': num_songs,
    'eval_results': eval_results,  # Tambahkan hasil evaluasi
    'data_splits': {  # Tambahkan informasi split
        'train_size': len(train_dataset),
        'val_size': len(val_dataset), 
        'test_size': len(test_dataset)
    }
}, 'songrec_model_alpha.pth')
print("✅ Model berhasil diekspor ke 'songrec_model_alpha.pth'")
print(f"📁 File berisi: model weights, vocab, mappings, konfigurasi, dan hasil evaluasi")
print(f"📊 Data splits - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"🎯 Test Accuracy: {eval_results['test_accuracy']:.4f} ({eval_results['test_accuracy']*100:.2f}%)")

Mengekspor model...
✅ Model berhasil diekspor ke 'songrec_model_alpha.pth'
📁 File berisi: model weights, vocab, mappings, konfigurasi, dan hasil evaluasi
📊 Data splits - Train: 29437, Val: 6308, Test: 6308
🎯 Test Accuracy: 0.5720 (57.20%)
